# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [33]:
print('Ready. No external packages required for this practical.')

Ready. No external packages required for this practical.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [34]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'i am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [35]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
None


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [36]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [37]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams      -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
The weather is fine today        -> Tell me more about that.


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [38]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
    (['i', 'feel', '?X'],    'What makes you feel {X}?'),
    (['i', 'love', '?X'],    'Why do you love {X}?'),
    (['i', 'hate', '?X'],    'What is it about {X} that bothers you?'),
    (['i', 'want', '?X'],   'What would having {X} change for you?'),
    (['my', 'teacher', 'is', '?X'], 'How does your teacher being {X} affect you?'),
    (['i', 'miss', '?X'],   'What do you miss most about {X}?'),
]

# Test each new rule with a matching sentence.
for sentence in [
    'I feel tired after class',
    'I love cricket',
    'I hate exams',
    'I want more free time',
    'My teacher is strict',
    'I miss my old school',
]:
    print(f'{sentence:30} -> {respond(sentence.lower())}')


I feel tired after class       -> What makes you feel tired after class?
I love cricket                 -> Why do you love cricket?
I hate exams                   -> What is it about exams that bothers you?
I want more free time          -> What would having more free time change for you?
My teacher is strict           -> How does your teacher being strict affect you?
I miss my old school           -> What do you miss most about your old school?


### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [39]:
def match(pattern, tokens):
    """Match a list of tokens against a pattern that may contain variables.
    A variable like '?X' captures all tokens that fit between fixed words.
    Returns a dict of bindings, or None if the pattern does not match."""
    bindings = {}

    def backtrack(p_index, t_index):
        if p_index == len(pattern):
            return t_index == len(tokens)

        p = pattern[p_index]

        if p.startswith('?'):
            # Try every possible length for the captured phrase.
            remaining = len(pattern) - p_index - 1
            max_len = len(tokens) - t_index - remaining

            for length in range(max_len + 1):
                captured = tokens[t_index:t_index + length]
                bindings[p] = captured
                if backtrack(p_index + 1, t_index + length):
                    return True
                del bindings[p]
            return False

        if t_index >= len(tokens) or tokens[t_index] != p:
            return False
        return backtrack(p_index + 1, t_index + 1)

    return bindings if backtrack(0, 0) else None

print(match(['i', '?X', 'about', 'exams'], 'i am worried about exams'.split()))
print(match(['i', '?X', 'about', 'exams'], 'i am worried about my exams'.split()))
print(match(['i', '?X', 'am', '?Y'], 'i feel really am happy'.split()))


{'?X': ['am', 'worried']}
None
{'?X': ['feel', 'really'], '?Y': ['happy']}


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [40]:
# Expression as nested list: 1 * x + 0 is ['+', ['*', 1, 'x'], 0]
expr = ['+', ['*', 1, 'x'], 0]

rules = [
    (['*', 1, '?X'], '?X'),
    (['*', '?X', 1], '?X'),
    (['+', '?X', 0], '?X'),
    (['+', 0, '?X'], '?X'),
]


def match(pattern, expression):
    """Match a symbolic pattern with wildcard variables like '?X'."""
    if isinstance(pattern, str) and pattern.startswith('?'):
        return {pattern: expression}

    if isinstance(pattern, list) and isinstance(expression, list):
        if len(pattern) != len(expression):
            return None
        bindings = {}
        for p, e in zip(pattern, expression):
            result = match(p, e)
            if result is None:
                return None
            for key, value in result.items():
                if key in bindings and bindings[key] != value:
                    return None
                bindings[key] = value
        return bindings

    if pattern == expression:
        return {}
    return None


def substitute(template, bindings):
    if isinstance(template, str) and template.startswith('?'):
        return bindings[template]
    if isinstance(template, list):
        return [substitute(item, bindings) for item in template]
    return template


def rewrite(node):
    if not isinstance(node, list):
        return node

    new_node = [rewrite(item) for item in node]
    for pattern, replacement in rules:
        bindings = match(pattern, new_node)
        if bindings is not None:
            return substitute(replacement, bindings)
    return new_node


def simplify(expr):
    current = expr
    while True:
        new_expr = rewrite(current)
        if new_expr == current:
            return new_expr
        current = new_expr


print('original :', expr)
print('simplified:', simplify(expr))


original : ['+', ['*', 1, 'x'], 0]
simplified: x


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [41]:
# Task 4: tester interaction
# A classmate used the responder without knowing the rules.
# Their feedback was recorded below.

conversation = [
    ("I am worried about my exams", "How long have you been worried about your exams?"),
    ("I need more time", "Why do you need more time?"),
    ("I feel tired", "What makes you feel tired?")
]

for question, answer in conversation:
    print(f"User: {question}\nBot: {answer}\n")

User: I am worried about my exams
Bot: How long have you been worried about your exams?

User: I need more time
Bot: Why do you need more time?

User: I feel tired
Bot: What makes you feel tired?



### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

### What the program actually stores

- A list of tokens for each sentence, such as `['i', 'am', 'worried', 'about', 'my', 'exams']`
- A dictionary of variable bindings like `{'?X': ['worried', 'about', 'my', 'exams']}`
- A set of rule patterns such as `['i', 'am', '?X']`
- A few fixed reply templates like `How long have you been {X}?`
- A small replacement table such as `{'my': 'your', 'i': 'you', 'am': 'are'}`

### What a user normally assumes it stores

- The program understands emotions, problems, and intentions.
- It can know whether a person is stressed, tired, or anxious.
- It stores real-world meaning, context, and beliefs about the user.
- It appears to understand the conversation as a person would.

### The gap

The system does not store meaning in the human sense. It only stores strings, token lists, pattern matches, and rule templates. It does not know what “stress,” “love,” or “exams” mean; it only sees word shapes and applies a matching strategy. The convincing reply comes from language patterns, not from real understanding.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| I am worried about my exams | ['i', 'am', '?X'] | How long have you been worried about your exams? | Yes |
| I need more time | ['i', 'need', '?X'] | Why do you need more time? | Yes |
| I feel tired | ['i', 'feel', '?X'] | What makes you feel tired? | Yes |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
| It understood stress and emotion, and that I was talking about real personal problems. | Only lowercase token lists, a small set of matching rules, and the captured words from the sentence. |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. The responder does not really understand the user in a human sense. It only stores tokens, patterns, variable bindings, and a few rewriting rules. The reply looks sensible because it matches the input shape and swaps a few words, but it has no idea what emotions or events actually mean.

2. Swapping person words changes the reply from first-person to second-person language, which makes it feel like a conversation with the user rather than a statement about them. The wording becomes natural and personal, so the response reads as though it is engaging with the speaker.

3. It shows that people often judge a conversational system by how it sounds and how fluent it appears, not by checking whether it truly understands. In other words, surface-level language can create the impression of understanding even when the system is only following rules and patterns.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [ ]:
NAME = "Qasim Talal"
ROLL_NUMBER = "2025-cse-022"
print(NAME, ROLL_NUMBER)

Qasim Talal 2025-cse-022
